# Machine Learning at Scale: Course Review

## Learning Objectives

1. **Summarize** the core algorithmic paradigms for large-scale ML
2. **Map** each section's algorithms to their scaling strategy
3. **Compare** algorithms across dimensions: passes, memory, accuracy
4. **Identify** when to apply each approach

## The Fundamental Challenges

Large-scale ML faces three core constraints:

| Constraint | Impact | Solution |
|------------|--------|---------|
| **Memory** | Can't hold all data in RAM | Stream processing, sufficient statistics, sketches |
| **Time** | Too many passes over disk | 1–2 pass algorithms, MapReduce parallelism |
| **Communication** | Distributed systems cost | Minimize shuffle; locality in MapReduce |

The algorithms in this course address these with one of three strategies:
1. **Summarize:** maintain a compact synopsis (sufficient stats, sketch, reservoir)
2. **Approximate:** accept controlled error for dramatic space/time savings
3. **Parallelize:** divide work across machines (MapReduce, Spark)

## Algorithm Summary by Section

### Section 001: Infrastructure
| Algorithm | Passes | Memory | Purpose |
|-----------|--------|--------|---------|
| MapReduce word count | 1 | $O(V)$ | Baseline |
| Matrix multiply (2-pass) | 2 | $O(n^2/k)$ | Scalable linear algebra |

### Section 002: Similarity
| Algorithm | Passes | Memory | Accuracy |
|-----------|--------|--------|---------|
| MinHash | 1 | $O(k \cdot |S|)$ | $±\sqrt{1/k}$ |
| LSH | 1 | $O(b \cdot r \cdot |S|)$ | Tunable FP/FN |

### Section 003: Link Analysis
| Algorithm | Passes | Memory | Note |
|-----------|--------|--------|------|
| PageRank (power iter.) | $O(\log(1/\epsilon))$ | $O(n)$ | Sparse matrix |
| Block-stripe PageRank | Same | Tunable | Web-scale |

### Section 004: Frequent Itemsets
| Algorithm | Passes | Memory | Error |
|-----------|--------|--------|-------|
| A-Priori | 2 per itemset size | $O(I^{k-1})$ | Exact |
| PCY | 2 | $O(\text{RAM})$ | Exact |
| SON | 2 | $O(\text{chunk})$ | Exact |

### Section 005: Data Streams
| Algorithm | Space | Passes | Error |
|-----------|-------|--------|-------|
| DGIM | $O(\log^2 N)$ | 1 | $\leq 50\%$ count |
| Bloom filter | $O(n)$ bits | 1 | One-sided FP |
| Flajolet-Martin | $O(\log m)$ | 1 | $O(1/\sqrt{k})$ |
| AMS | $O(k)$ | 1 | $(1\pm\epsilon)$ $F_2$ |
| Count-Min | $O(\frac{1}{\epsilon}\log\frac{1}{\delta})$ | 1 | Additive $\epsilon n$ |
| Reservoir | $O(k)$ | 1 | Exactly uniform |

### Sections 006–007: Clustering & Dimensionality Reduction
| Algorithm | Passes | Memory | Note |
|-----------|--------|--------|------|
| BFR (k-means at scale) | 1 per chunk | $O(k)$ suf. stats | Disk-resident |
| CURE | 2 | $O(n \cdot \text{reps})$ | Non-spherical clusters |
| SVD truncated | — | $O(md + nd)$ | Eckart-Young optimal |
| CUR decomposition | 2 | $O(c + r)$ | Sparse + interpretable |
| Random projection | 1 | $O(d \cdot k)$ | JL lemma, $O(\log n)$ dim |

### Section 008: Recommender Systems
| Algorithm | Training | Space | Error |
|-----------|----------|-------|-------|
| User-user CF | $O(m^2 n)$ | $O(m^2)$ | High variance |
| Item-item CF | $O(n^2 m)$ | $O(n^2)$ | Stable |
| Matrix factorization | $O(|\Omega| k)$ | $O((m+n)k)$ | Best RMSE |

In [ ]:
# Summary: trade-offs across algorithms
algorithms = [
    # (Name, Passes, Memory, Passes_approx, Error_type)
    ("MinHash (k=100)",      "1",     "O(k|S|)",     "Exact hash",  "±√(1/k) in Jaccard"),
    ("LSH (b×r=10×10)",     "1",     "O(br|S|)",    "Exact hash",  "Tunable FP/FN"),
    ("PageRank",             "O(logε)","O(n)",        "Sparse",     "Exact (converges)"),
    ("Bloom filter",         "1",     "O(n) bits",   "Hash array", "FP only, no FN"),
    ("Count-Min",            "1",     "O(1/ε log 1/δ)","Hash grid", "Additive εn"),
    ("Reservoir sample",     "1",     "O(k)",        "Random",     "Exactly uniform"),
    ("Truncated SVD",        "—",     "O(mk+nk)",    "Dense",      "Optimal rank-k"),
    ("Random projection",    "1",     "O(dk)",       "Random",     "JL: (1±ε) dist"),
    ("TransE",               "Many",  "O((|E|+|R|)d)","Gradient", "Approx KG embedding"),
]
print(f"{'Algorithm':<25} {'Passes':<10} {'Memory':<22} {'Error'}")
print("-"*80)
for name, passes, mem, typ, err in algorithms:
    print(f"{name:<25} {passes:<10} {mem:<22} {err}")

## Key Takeaways

### When to use streaming algorithms
- Data arrives faster than you can store it
- Memory is the binding constraint
- Accept approximate answers with provable bounds

### When to use MapReduce / distributed computing
- Data fits on disk but not RAM
- Task is naturally parallelizable (embarrassingly parallel > shuffle-heavy)
- Multiple passes are acceptable

### When to use approximation
- Exact answer requires $O(n^2)$ or $O(n^3)$ — unacceptable at scale
- $(1\pm\epsilon)$ approximation is sufficient for the application
- JL, sketching, sampling all trade accuracy for space/time

### When to use model-based methods (SVD, MF, GNNs)
- Data has low-rank structure (user-item ratings, graph adjacency)
- Features and relations both matter
- Can afford offline training; need fast online inference

**The common thread:** every large-scale ML algorithm exploits structure — sparsity, low rank, subadditivity, locality, or hash functions — to escape the curse of dimensionality.